# Lumpy 03: Results Review

This notebook reviews the final outputs written by `Lumpy 02: Streamlined Forecasting`.


## 1. Load the final result

We load the headline scorecard, segment results, block forecasts and SKU delivery table from the new 02 output folders.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, start.parent, start.parent.parent, start / "Inchscape Repo"]:
        if (candidate / "src" / "lumpy_forecasting.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the Lumpy Fellas project root.")


project_root = find_project_root()
result_dir = project_root / "results/lumpy_02_forecasting"
review_dir = project_root / "docs/lumpy_02_forecasting"

headline = pd.read_csv(review_dir / "lumpy_02_headline_scorecard.csv")
segments = pd.read_csv(review_dir / "lumpy_02_segment_scorecard.csv")
reliability = pd.read_csv(review_dir / "lumpy_02_reliability_summary.csv")
forecasts = pd.read_csv(result_dir / "lumpy_02_final_690_block_forecasts.csv", parse_dates=["block_start"])
delivery = pd.read_csv(result_dir / "lumpy_02_all_690_sku_delivery.csv")

print("Project root:", project_root)
print("Forecast rows:", len(forecasts))
print("Delivery rows:", len(delivery))


## 2. Headline scorecard

We compare the full governance population with the actionable portfolio used for forecast-led planning.


In [ ]:
headline_view = headline[[
    "scorecard", "all_skus", "positive_skus", "under_50", "under_70", "under_100",
    "pct_below_50", "pct_below_70", "pct_below_100", "median_wmape", "portfolio_wmape", "bias_pct"
]].copy()
display(headline_view.round(2))


## 3. Forecast against actual demand

We aggregate the six official blocks to show the level and direction of the remaining portfolio bias.


In [ ]:
block_totals = (
    forecasts.groupby(["block_number", "block_start"], as_index=False)
    .agg(actual=("target", "sum"), forecast=("forecast", "sum"))
    .sort_values("block_number")
)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(block_totals.block_start, block_totals.actual, marker="o", color="black", label="Actual")
ax.plot(block_totals.block_start, block_totals.forecast, marker="o", color="#2c7fb8", label="Forecast")
ax.set_title("Final 690-SKU forecast by three-month block")
ax.set_ylabel("Demand units")
ax.grid(alpha=0.25)
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

display(block_totals.round(2))


## 4. Segment performance

We separate the stronger point-forecast segments from the areas that remain policy or manual-review problems.


In [ ]:
segment_view = segments[[
    "segment", "all_skus", "positive_skus", "under_50", "under_70", "under_100",
    "pct_below_50", "pct_below_70", "pct_below_100", "median_wmape", "portfolio_wmape", "bias_pct"
]].sort_values("pct_below_70", ascending=False)
display(segment_view.round(2))

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_data = segment_view.sort_values("pct_below_70")
ax.barh(plot_data.segment, plot_data.pct_below_70, color="#2c7fb8")
ax.axvline(50, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Positive SKUs below 70% WMAPE (%)")
ax.set_title("Below-70 coverage by segment")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Reliability and exceptions

We show how many SKUs can use the forecast directly and which SKUs need review or exception handling.


In [ ]:
display(reliability.sort_values("skus", ascending=False))

exceptions = delivery.loc[
    delivery.reliability_tier.isin(["manual_review_with_forecast", "exception_policy"])
].copy()
exceptions["underforecast_units"] = (exceptions.actual_18m - exceptions.forecast_18m).clip(lower=0)
top_exceptions = exceptions.sort_values("underforecast_units", ascending=False).head(25)

display(top_exceptions[[
    "sku_id", "segment", "reliability_tier", "actual_18m", "forecast_18m",
    "underforecast_units", "wmape", "champion_source"
]].round(2))
